# Task 2 Playground: NumPy MLP and Training

This notebook explores the Task 2 implementation in `nn/` and `train/trainer.py`: initialization, parameters/modules, dense layers, activation modules, `Sequential`, AdamW updates, binary masks and optimizer state, global gradient clipping, and mini-batch classifier training.

Run cells from top to bottom. Open the notebook from the repository root and select the project `.venv` Python kernel so `nn`, `autodiff`, and `train` import correctly. The examples use deterministic seeds and NumPy only. The repository does **not** implement Task 3/4 pruning schedules, regrowth policies, or Pareto sweeps.

In [3]:
import numpy as np

from autodiff import Tensor, mean, relu, softmax, softmax_cross_entropy, sum
from nn import (
    Adam, Linear, Module, Parameter, ReLU, Sequential, Sigmoid, Tanh,
    he_normal, xavier_uniform,
)
from train.trainer import EpochMetrics, clip_grad_norm, train_classifier

np.set_printoptions(precision=4, suppress=True)
print("Task 2 playground is ready")

Task 2 playground is ready


## 1. Weight initialization

`he_normal((fan_in, fan_out), rng)` samples with standard deviation `sqrt(2 / fan_in)`. `xavier_uniform(...)` samples from `[-sqrt(6 / (fan_in + fan_out)), +sqrt(...)]`. Passing a seeded NumPy generator makes initialization reproducible.

In [4]:
rng_a = np.random.default_rng(1)
rng_b = np.random.default_rng(17)
he_weights = he_normal((4, 3), rng_a)
he_repeat = he_normal((4, 3), rng_b)

he_weights

array([[ 0.2444,  0.581 ,  0.2337],
       [-0.9215,  0.6402,  0.3156],
       [-0.3797,  0.4109,  0.2578],
       [ 0.208 ,  0.0201,  0.3866]])

In [5]:
rng_a = np.random.default_rng(17)
rng_b = np.random.default_rng(17)
he_weights = he_normal((4, 3), rng_a)
he_repeat = he_normal((4, 3), rng_b)

rng_c = np.random.default_rng(17)
xavier_weights = xavier_uniform((4, 3), rng_c)
xavier_bound = np.sqrt(6.0 / (4 + 3))

print("He weights:\n", he_weights)
print("He sample std (small sample):", he_weights.std())
print("same seed gives identical He weights:", np.array_equal(he_weights, he_repeat))
print("Xavier weights:\n", xavier_weights)
print("expected Xavier bound:", xavier_bound)
print("all Xavier values inside bound:", np.all(np.abs(xavier_weights) <= xavier_bound))

He weights:
 [[ 0.7787  0.2393 -0.3818]
 [-0.8911 -1.3397  0.0132]
 [-0.5732 -0.6167 -0.157 ]
 [-0.0367 -1.6099  0.6542]]
He sample std (small sample): 0.6995339627921183
same seed gives identical He weights: True
Xavier weights:
 [[ 0.639  -0.6278  0.1069]
 [-0.2443 -0.5278 -0.2114]
 [-0.133   0.2062  0.4377]
 [-0.8975 -0.4554  0.1928]]
expected Xavier bound: 0.9258200997725514
all Xavier values inside bound: True


`Linear` selects He initialization when its `activation` is `"relu"`; `"tanh"`, `"sigmoid"`, and `"linear"` select Xavier. The string chooses initialization only—the activation itself is a separate module/function.

In [6]:
for activation_name in ("relu", "tanh", "sigmoid", "linear"):
    layer = Linear(4, 3, activation=activation_name, rng=np.random.default_rng(5))
    print(f"{activation_name:7} W shape={layer.W.shape}, bias={layer.b.data}, std={layer.W.data.std():.4f}")

relu    W shape=(4, 3), bias=[0. 0. 0.], std=0.6343
tanh    W shape=(4, 3), bias=[0. 0. 0.], std=0.5757
sigmoid W shape=(4, 3), bias=[0. 0. 0.], std=0.5757
linear  W shape=(4, 3), bias=[0. 0. 0.], std=0.5757


## 2. Parameters and modules

`Parameter` is a trainable leaf `Tensor` (`requires_grad=True`). `Module` provides callable-forward behavior, recursive parameter collection, gradient clearing, and optional parameter masks. `Linear` owns a weight matrix `W` and a zero-initialized bias vector `b`.

In [7]:
standalone_parameter = Parameter([[1.0, -1.0], [0.5, 2.0]])
print("Parameter is a Tensor:", isinstance(standalone_parameter, Tensor))
print("requires_grad:", standalone_parameter.requires_grad)
print("gradient before backward:", standalone_parameter.grad)

Parameter is a Tensor: True
requires_grad: True
gradient before backward: None


In [8]:
# standalone_parameter = Parameter([[1.0, -1.0], [0.5, 2.0]])
# print("Parameter is a Tensor:", isinstance(standalone_parameter, Tensor))
# print("requires_grad:", standalone_parameter.requires_grad)
# print("gradient before backward:", standalone_parameter.grad)

linear = Linear(2, 3, activation="relu", rng=np.random.default_rng(3))
print("weight shape:", linear.W.shape, "bias shape:", linear.b.shape)
print("bias starts at:", linear.b.data)
print("parameters:", linear.parameters())
print("initial weight mask:\n", linear.mask)
print("parameter_masks mapping:", linear.parameter_masks())

weight shape: (2, 3) bias shape: (3,)
bias starts at: [0. 0. 0.]
parameters: [Tensor(data=array([[ 2.0409, -2.5557,  0.4181],
       [-0.5678, -0.4526, -0.2156]]), requires_grad=True, grad=None), Tensor(data=array([0., 0., 0.]), requires_grad=True, grad=None)]
initial weight mask:
 [[1. 1. 1.]
 [1. 1. 1.]]
parameter_masks mapping: {Tensor(data=array([[ 2.0409, -2.5557,  0.4181],
       [-0.5678, -0.4526, -0.2156]]), requires_grad=True, grad=None): array([[1., 1., 1.],
       [1., 1., 1.]])}


A module can recursively discover parameters from nested child modules. Parameters are returned once, in traversal order, and `zero_grad()` clears parameter gradients without changing parameter values.

In [9]:
first = Linear(2, 4, rng=np.random.default_rng(1))
last = Linear(4, 2, activation="linear", rng=np.random.default_rng(2))
model = Sequential(first, ReLU(), last)

In [10]:
model.modules

(<nn.layers.Linear at 0x1875d895cd0>,
 <nn.layers.Linear at 0x1875d8b8710>)

In [11]:
model.parameters

<bound method Module.parameters of <nn.layers.Sequential object at 0x000001875D8BAC60>>

In [12]:
print("Sequential modules:", model.modules)
print("all trainable parameters in order:")
for index, parameter in enumerate(model.parameters()):
    print(index, parameter.shape)
print("number of parameters:", len(model.parameters()))
print("weight mask entries:", len(model.parameter_masks()))

for parameter in model.parameters():
    parameter.grad = np.ones_like(parameter.data)
model.zero_grad()
print("all parameter gradients cleared:", all(p.grad is None for p in model.parameters()))

Sequential modules: (<nn.layers.Linear object at 0x000001875D895CD0>, <nn.layers.ReLU object at 0x0000018747515DF0>, <nn.layers.Linear object at 0x000001875D8B8710>)
all trainable parameters in order:
0 (2, 4)
1 (4,)
2 (4, 2)
3 (2,)
number of parameters: 4
weight mask entries: 2
all parameter gradients cleared: True


## 3. Linear layer forward and backward

A `Linear(in_features, out_features)` layer computes `inputs @ W + b` and accepts any input with `in_features` as its last dimension. Inputs need not request gradients for the weights and bias to receive gradients.

In [13]:
layer = Linear(2, 2, activation="linear", rng=np.random.default_rng(0))
layer.W.data[...] = [[2.0, -1.0], [0.5, 3.0]]
layer.b.data[...] = [0.25, -0.5]
batch = Tensor([[1.0, 2.0], [-1.0, 0.5]])

output = layer(batch)
print("output:\n", output.data)
sum(output).backward()
print("W gradient:\n", layer.W.grad)
print("b gradient:", layer.b.grad)
print("input gradient:", batch.grad)

output:
 [[ 3.25  4.5 ]
 [-1.5   2.  ]]
W gradient:
 [[0.  0. ]
 [2.5 2.5]]
b gradient: [2. 2.]
input gradient: None


The Linear mask applies to `W` in the forward pass and routes no gradient to masked weight entries. The mask must be binary and broadcastable to the weight shape.

In [14]:
masked_layer = Linear(2, 2, activation="linear", rng=np.random.default_rng(0))
masked_layer.W.data[...] = [[1.0, 2.0], [3.0, 4.0]]
masked_layer.mask[...] = [[1.0, 0.0], [1.0, 1.0]]
masked_out = masked_layer(np.array([[2.0, 5.0]]))
print("masked forward output:", masked_out.data)
sum(masked_out).backward()
print("weight gradient (masked coordinate is zero):\n", masked_layer.W.grad)
print("reported mask:", masked_layer.parameter_masks()[masked_layer.W])

masked forward output: [[17. 20.]]
weight gradient (masked coordinate is zero):
 [[2. 0.]
 [5. 5.]]
reported mask: [[1. 0.]
 [1. 1.]]


## 4. Activation modules and Sequential composition

The module wrappers `ReLU`, `Tanh`, and `Sigmoid` reuse the corresponding Task 1 autodiff operations. `Sequential` calls its modules from first to last and collects all child parameters.

In [15]:
activation_values = np.array([[-2.0, -0.5, 0.0, 0.5, 2.0]])
for activation_module in (ReLU(), Tanh(), Sigmoid()):
    result = activation_module(activation_values)
    print(type(activation_module).__name__, "->", result.data)

two_layer = Sequential(
    Linear(2, 3, activation="relu", rng=np.random.default_rng(7)),
    ReLU(),
    Linear(3, 2, activation="linear", rng=np.random.default_rng(8)),
)
two_layer_logits = two_layer(np.array([[1.0, -1.0], [0.5, 2.0]]))
print("Sequential output shape:", two_layer_logits.shape)
print("Sequential logits:\n", two_layer_logits.data)
print("parameter count:", len(two_layer.parameters()))

ReLU -> [[0.  0.  0.  0.5 2. ]]
Tanh -> [[-0.964  -0.4621  0.      0.4621  0.964 ]]
Sigmoid -> [[0.1192 0.3775 0.5    0.6225 0.8808]]
Sequential output shape: (2, 2)
Sequential logits:
 [[-0.0559  1.2572]
 [ 0.      0.    ]]
parameter count: 4


A classification model returns raw scores (logits), not softmax probabilities. `softmax_cross_entropy` performs the stable softmax/loss calculation; you can call `softmax(logits)` for readable probabilities.

In [16]:
class_targets = np.array([[1.0, 0.0], [0.0, 1.0]])
class_loss = softmax_cross_entropy(two_layer_logits, class_targets)
class_loss.backward()
print("probabilities:\n", softmax(two_layer_logits).data)
print("mean loss:", class_loss.data.item())
print("first parameter gradient shape:", two_layer.parameters()[0].grad.shape)

probabilities:
 [[0.212 0.788]
 [0.5   0.5  ]]
mean loss: 1.122198445723348
first parameter gradient shape: (2, 3)


## 5. Adam with decoupled weight decay

`Adam(parameters, lr, beta1, beta2, eps, weight_decay)` stores first and second moments (`m`, `v`) and a step counter (`t`). `step()` updates parameters with bias-corrected moments and decoupled weight decay. `zero_grad()` clears gradients between batches.

In [17]:
parameter = Parameter([1.0, -2.0])
parameter.grad = np.array([2.0, -4.0])
optimizer = Adam([parameter], lr=0.1, beta1=0.9, beta2=0.999, eps=1e-8, weight_decay=0.2)
print("before: data", parameter.data, "m", optimizer.m[0], "v", optimizer.v[0], "t", optimizer.t)

optimizer.step()
print("after one step: data", parameter.data)
print("first moment:", optimizer.m[0])
print("second moment:", optimizer.v[0])
print("step counter:", optimizer.t)
optimizer.zero_grad()
print("gradient after optimizer.zero_grad():", parameter.grad)

before: data [ 1. -2.] m [0. 0.] v [0. 0.] t 0
after one step: data [ 0.88 -1.86]
first moment: [ 0.2 -0.4]
second moment: [0.004 0.016]
step counter: 1
gradient after optimizer.zero_grad(): None


On the first step, bias correction makes the normalized Adam update approximately `sign(gradient)`. Decoupled decay separately multiplies the current parameter by `1 - lr * weight_decay` before the adaptive update. Try changing the learning rate, betas, epsilon, or decay above.

In [18]:
# Show that an optimizer can also be created for all parameters in a module.
optimizer_for_model = Adam(two_layer.parameters(), lr=0.01, weight_decay=1e-4)
print("optimizer parameter count:", len(optimizer_for_model.params))
print("moment shapes:", [moment.shape for moment in optimizer_for_model.m])

optimizer parameter count: 4
moment shapes: [(2, 3), (3,), (3, 2), (2,)]


## 6. Optimizer masks, clearing moments, and re-enabling connections

Masks passed to `Adam.step` are keyed by **parameter index**. Inactive entries are set to zero and their moments are cleared. If later re-enabled, they start with fresh moments rather than stale momentum. This is mask/optimizer-state correctness—not an automated pruning or regrowth scheduler.

In [19]:
masked_parameter = Parameter([1.0, 1.0])
masked_optimizer = Adam([masked_parameter], lr=0.1)

# First update: both coordinates are active, so both acquire moment history.
masked_parameter.grad = np.array([1.0, 1.0])
masked_optimizer.step()
print("initial m:", masked_optimizer.m[0])
print("initial v:", masked_optimizer.v[0])

# Freeze the second coordinate. Its value and both moments are zeroed.
masked_parameter.grad = np.array([2.0, 100.0])
masked_optimizer.step(masks={0: np.array([1.0, 0.0])})
print("after masking: data", masked_parameter.data)
print("after masking: m", masked_optimizer.m[0], "v", masked_optimizer.v[0])

# Re-enable it: its first moment is just (1-beta1) * this new gradient.
masked_parameter.grad = np.array([1.0, 3.0])
masked_optimizer.step(masks={0: np.array([1.0, 1.0])})
print("after re-enable: data", masked_parameter.data)
print("after re-enable: m", masked_optimizer.m[0])
print("expected fresh second-coordinate m:", (1 - masked_optimizer.beta1) * 3.0)

initial m: [0.1 0.1]
initial v: [0.001 0.001]
after masking: data [0.8035 0.    ]
after masking: m [0.29 0.  ] v [0.005 0.   ]
after re-enable: data [ 0.7093 -0.1   ]
after re-enable: m [0.361 0.3  ]
expected fresh second-coordinate m: 0.29999999999999993


.Masking is also enforced when a parameter has no gradient: inactive values and moments are still zeroed. Mask arrays must contain only zeroes and ones and broadcast to the parameter shape.

In [20]:
no_grad_parameter = Parameter([4.0, -3.0])
no_grad_optimizer = Adam([no_grad_parameter])
no_grad_optimizer.step(masks={0: np.array([1.0, 0.0])})
print("masked without gradient:", no_grad_parameter.data)
print("moments:", no_grad_optimizer.m[0], no_grad_optimizer.v[0])

masked without gradient: [4. 0.]
moments: [0. 0.] [0. 0.]


## 7. Global gradient norm clipping

`clip_grad_norm(parameters, max_norm)` measures all present gradients together, rescales them only when the combined norm exceeds the limit, and returns the original norm.

In [21]:
clip_a = Parameter([0.0, 0.0])
clip_b = Parameter([0.0])
clip_a.grad = np.array([3.0, 4.0])
clip_b.grad = np.array([12.0])
before_norm = np.sqrt(np.sum(clip_a.grad**2) + np.sum(clip_b.grad**2))
returned_norm = clip_grad_norm([clip_a, clip_b], max_norm=5.0)
after_norm = np.sqrt(np.sum(clip_a.grad**2) + np.sum(clip_b.grad**2))
print("original norm:", before_norm)
print("function returned:", returned_norm)
print("clipped gradients:", clip_a.grad, clip_b.grad)
print("combined norm after clipping:", after_norm)

original norm: 13.0
function returned: 13.0
clipped gradients: [1.1538 1.5385] [4.6154]
combined norm after clipping: 4.999999999999615


If the norm is already under the limit, gradients are left unchanged. The training loop applies clipping after backpropagation and before every optimizer update.

In [15]:
small_gradient = Parameter([0.3, 0.4])
small_gradient.grad = np.array([0.3, 0.4])
small_before = small_gradient.grad.copy()
small_norm = clip_grad_norm([small_gradient], max_norm=1.0)
print("norm below cap:", small_norm)
print("unchanged:", np.array_equal(small_gradient.grad, small_before))

norm below cap: 0.5
unchanged: True


## 8. Make a deterministic three-class dataset

The trainer accepts a 2D feature array and a 1D integer label array containing class indices. Here we create three noisy clusters without external datasets.

In [22]:
data_rng = np.random.default_rng(21)
centers = np.array([[-2.0, -1.0], [2.0, -1.0], [0.0, 2.0]])
features = np.concatenate([
    center + data_rng.normal(0.0, 0.5, size=(60, 2))
    for center in centers
])
labels = np.repeat(np.arange(3), 60)
permutation = data_rng.permutation(len(labels))
features, labels = features[permutation], labels[permutation]

print("features shape:", features.shape, "labels shape:", labels.shape)
print("class counts:", np.bincount(labels))
print("first five features:\n", features[:5])
print("first five labels:", labels[:5])

features shape: (180, 2) labels shape: (180,)
class counts: [60 60 60]
first five features:
 [[-0.1281  2.0618]
 [ 2.011  -0.7107]
 [-1.9607 -1.0986]
 [-2.9476 -1.1928]
 [-0.1321  2.8344]]
first five labels: [2 1 0 0 2]


## 9. Train a classifier with mini-batches

`train_classifier(model, optimizer, inputs, labels, epochs, batch_size, seed, max_grad_norm)` shuffles the data each epoch, includes the final incomplete batch, converts class indices to one-hot targets, calculates mean cross-entropy, clips the global gradient norm, and calls Adam. It returns one `EpochMetrics(epoch, loss, accuracy)` record per epoch.

The optimizer must contain exactly the model's parameters. The model must return a 2D array of at least two class scores per example.

In [24]:
model_rng = np.random.default_rng(7)
classifier = Sequential(
    Linear(2, 12, activation="relu", rng=model_rng),
    ReLU(),
    Linear(12, 3, activation="linear", rng=model_rng),
)
classifier_optimizer = Adam(classifier.parameters(), lr=0.03, weight_decay=1e-4)

history = train_classifier(
    classifier,
    classifier_optimizer,
    features,
    labels,
    epochs=15,
    batch_size=32,
    seed=11,
    max_grad_norm=5.0,
)

print("history record type:", type(history[0]).__name__)
print("first epoch:", history[0])
print("last epoch:", history[-1])
print("optimizer updates:", classifier_optimizer.t)
print("first/last loss:", history[0].loss, history[-1].loss)
print("first/last accuracy:", history[0].accuracy, history[-1].accuracy)

history record type: EpochMetrics
first epoch: EpochMetrics(epoch=1, loss=1.5326972214242671, accuracy=0.23333333333333334)
last epoch: EpochMetrics(epoch=15, loss=0.001314935434007753, accuracy=1.0)
optimizer updates: 90
first/last loss: 1.5326972214242671 0.001314935434007753
first/last accuracy: 0.23333333333333334 1.0


The number of batches per epoch is the ceiling of `sample_count / batch_size`; no samples are dropped. With 180 samples and batch size 32, each epoch has six updates (the last batch has 20 examples). Try changing the batch size and compare the optimizer step count.

In [25]:
samples, batch_size, epochs = len(features), 32, len(history)
batches_per_epoch = (samples + batch_size - 1) // batch_size
print("samples:", samples)
print("batch sizes:", [len(range(start, min(start + batch_size, samples))) for start in range(0, samples, batch_size)])
print("expected optimizer steps:", batches_per_epoch * epochs)
print("actual optimizer steps:", classifier_optimizer.t)

samples: 180
batch sizes: [32, 32, 32, 32, 32, 20]
expected optimizer steps: 90
actual optimizer steps: 90


Reproducibility requires both a fixed model-initialization seed and the same training shuffle seed. This small repeat uses freshly initialized identical models and identical training options.

In [26]:
def make_classifier(seed):
    rng = np.random.default_rng(seed)
    return Sequential(
        Linear(2, 8, activation="relu", rng=rng),
        ReLU(),
        Linear(8, 3, activation="linear", rng=rng),
    )

model_a, model_b = make_classifier(42), make_classifier(42)
history_a = train_classifier(model_a, Adam(model_a.parameters(), lr=0.02), features, labels,
                             epochs=3, batch_size=40, seed=99)
history_b = train_classifier(model_b, Adam(model_b.parameters(), lr=0.02), features, labels,
                             epochs=3, batch_size=40, seed=99)

metrics_a = np.array([[item.loss, item.accuracy] for item in history_a])
metrics_b = np.array([[item.loss, item.accuracy] for item in history_b])
print("identical metrics:", np.array_equal(metrics_a, metrics_b))
print("identical final weights:", all(
    np.array_equal(a.data, b.data)
    for a, b in zip(model_a.parameters(), model_b.parameters())
))

identical metrics: True
identical final weights: True


## 10. Masked training through the full loop

The trainer reads each Linear layer's parameter mask, maps it to the corresponding optimizer parameter index, and forwards it to `Adam.step`. This couples zeroed weights with cleared optimizer moments throughout training.

In [20]:
masked_model = make_classifier(6)
first_dense = masked_model.modules[0]
first_dense.mask[0, 0] = 0.0
masked_model_optimizer = Adam(masked_model.parameters(), lr=0.02)

masked_history = train_classifier(
    masked_model, masked_model_optimizer, features, labels,
    epochs=2, batch_size=45, seed=3,
)
weight_parameter_index = masked_model.parameters().index(first_dense.W)
print("masked weight after training:", first_dense.W.data[0, 0])
print("masked first moment:", masked_model_optimizer.m[weight_parameter_index][0, 0])
print("masked second moment:", masked_model_optimizer.v[weight_parameter_index][0, 0])
print("history:", masked_history)

masked weight after training: 0.0
masked first moment: 0.0
masked second moment: 0.0
history: [EpochMetrics(epoch=1, loss=1.8492658883539241, accuracy=0.6333333333333333), EpochMetrics(epoch=2, loss=1.0604254868298064, accuracy=0.65)]


## 11. Explore validation behavior

These compact examples intentionally pass invalid inputs so you can see how the APIs report common mistakes. Remove the `try/except` wrappers if you want to inspect the full traceback.

In [27]:
def run_bad_label_training():
    bad_label_model = make_classifier(1)
    bad_label_optimizer = Adam(bad_label_model.parameters())
    return train_classifier(
        bad_label_model, bad_label_optimizer, features[:4],
        labels[:4].astype(float), epochs=1,
    )

examples = [
    ("Linear wrong input width", lambda: Linear(3, 2)(np.ones((4, 2)))),
    ("unsupported Linear initializer name", lambda: Linear(2, 3, activation="softmax")),
    ("Adam needs a Parameter", lambda: Adam([Tensor([1.0], requires_grad=True)])),
    ("Sequential needs modules", lambda: Sequential(lambda x: x)),
    ("trainer needs integer labels", run_bad_label_training),
]
for description, operation in examples:
    try:
        operation()
    except (TypeError, ValueError) as error:
        print(f"{description}: {type(error).__name__}: {error}")

Linear wrong input width: ValueError: Expected input's last dimension to be 3; received shape (4, 2)
unsupported Linear initializer name: ValueError: activation must be 'relu', 'tanh', 'sigmoid', or 'linear'
Adam needs a Parameter: TypeError: Adam parameters must be Parameter instances
Sequential needs modules: TypeError: Sequential accepts only Module instances
trainer needs integer labels: ValueError: labels must contain integer class indices


## 12. Your Task 2 practice prompts

1. Change the hidden width, add a second hidden layer, and compare convergence.
2. Compare He and Xavier distributions with larger matrices; inspect empirical means and variances.
3. Try `Tanh()` or `Sigmoid()` in place of `ReLU()` and compare the loss/accuracy history.
4. Change `lr`, `beta1`, `beta2`, `eps`, and `weight_decay` one at a time. Inspect `t`, `m`, and `v`.
5. Change `batch_size` to a value that does not divide the dataset size. Verify the final partial batch contributes to every epoch and to the update count.
6. Mask a row/column of a weight matrix, train, and check that inactive values and optimizer moments remain zero.
7. Change `max_grad_norm` and inspect gradients immediately before and after `clip_grad_norm`.
8. For a custom training loop, use this order per batch: `optimizer.zero_grad()`, forward pass, scalar loss, `loss.backward()`, `clip_grad_norm(...)`, then `optimizer.step(masks=...)`.